In [1]:
%pip cache purge
%pip install -r ../requirements.txt


Files removed: 0 (0 bytes)
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.python.org/simple
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os


In [4]:
DIRECTORIES = [
    "../models", 
    # "../data/raw/files",
	"../plots",
    "../tmp",
    "../results",
]
deleted_files = []

for directory in DIRECTORIES:
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        continue

    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)

        if item == ".gitkeep":
            continue  # Skip .gitkeep

        if os.path.isfile(item_path):
            os.remove(item_path)
            deleted_files.append(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            deleted_files.append(item_path + "/")  

if deleted_files:
    print("Deleted files and directories:")
    for file in deleted_files:
        print(f" - {file}")
else:
    print("No files to delete.")


Deleted files and directories:
 - ../tmp/evens
 - ../tmp/events
 - ../tmp/events_ids
 - ../tmp/originalRaw
 - ../tmp/picks
 - ../tmp/raw


In [5]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import mne
from mne.io import concatenate_raws
from mne.io.edf import read_raw_edf
from mne.datasets import eegbci
from mne import events_from_annotations, pick_types
from mne.channels import make_standard_montage
from mne.preprocessing import ICA
from mne.decoding import SPoC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.base import BaseEstimator, TransformerMixin
from scipy import linalg
from joblib import dump, load
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import ShuffleSplit, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import concurrent.futures
import json
import hashlib
import os
import time

# Configuración global
matplotlib.use('TkAgg')
mne.set_log_level("CRITICAL")
DATA_SAMPLE_PATH = "../data/raw/files"

# Enumeraciones
# SUBJECTS = [1,2,3,4,5,6]
MODES = ["train", "predict", "all"]
TRANSFORMERS = ["FAST_CSP", "CSP", "SPoC"]

EXPERIMENTS = {
    "hands_vs_feet__action": ['do/hands', 'do/feet'],
    "hands_vs_feet__imagery": ['imagine/hands', 'imagine/feet'],
    "imagery_vs_action__hands": ['do/hands', 'imagine/hands'],
    "imagery_vs_action__feets": ['do/feet', 'imagine/feet'],
}

EXPERIMENTS_IDS = {
    'action': [5, 9, 13],
    'imagery': [6, 10, 14]
}

# Implementación de CSP
class CSP(BaseEstimator, TransformerMixin):
    """
    CSP implementation based on MNE implementation
    """
    def __init__(self, n_components=4):
        self.n_components = n_components
        self.filters = None
        self.n_classes = None
        self.mean = None
        self.std = None

    def calculate_cov_(self, X, y):
        """Calculate the covariance matrices for each class."""
        _, n_channels, _ = X.shape
        covs = []

        for l in self.n_classes:
            lX = X[np.where(y == l)]
            lX = lX.transpose([1, 0, 2])
            lX = lX.reshape(n_channels, -1)
            covs.append(np.cov(lX))

        return np.asarray(covs)

    def calculate_eig_(self, covs):
        """Calculate eigenvalues and eigenvectors for pairwise combinations of covariance matrices."""
        eigenvalues, eigenvectors = [], []

        for idx, cov in enumerate(covs):
            for iidx, compCov in enumerate(covs):
                if idx < iidx:
                    eigVals, eigVects = linalg.eig(cov, cov + compCov)
                    sorted_indices = np.argsort(np.abs(eigVals - 0.5))[::-1]
                    eigenvalues.append(eigVals[sorted_indices])
                    eigenvectors.append(eigVects[:, sorted_indices])

        return eigenvalues, eigenvectors

    def pick_filters(self, eigenvectors):
        """Select CSP filters based on the sorted eigenvectors."""
        filters = []

        for EigVects in eigenvectors:
            if filters == []:
                filters = EigVects[:, :self.n_components]
            else:
                filters = np.concatenate([filters, EigVects[:, :self.n_components]], axis=1)

        self.filters = filters.T

    def fit(self, X, y):
        self.n_classes = np.unique(y)

        if len(self.n_classes) < 2:
            raise ValueError("n_classes must be >= 2")

        covs = self.calculate_cov_(X, y)
        eigenvalues, eigenvectors = self.calculate_eig_(covs)
        self.pick_filters(eigenvectors)

        X = np.asarray([np.dot(self.filters, epoch) for epoch in X])
        X = (X ** 2).mean(axis=2)

        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0)

    def transform(self, X):
        X = np.asarray([np.dot(self.filters, epoch) for epoch in X])
        X = (X ** 2).mean(axis=2)
        X -= self.mean
        X /= self.std
        return X

    def fit_transform(self, X, y):
        self.fit(X, y)
        return self.transform(X)

# Funciones para procesamiento de datos
def fetch_data(subjNumber):
    run_execution = [5, 9, 13]  # (open and close both fists or both feet)
    run_imagery =  [6, 10, 14]  # (imagine opening and closing both fists or both feet)

    raw_files = []

    for i, j in zip(run_execution, run_imagery):
        try:
            # Datos de ejecución
            # Corregido: Solo pasamos el subject_id y run, y luego el path como parámetro nombrado
            raw_files_execution = [read_raw_edf(f, preload=True, stim_channel='auto') for f in
                                eegbci.load_data(subjNumber, i, path=DATA_SAMPLE_PATH)]
            raw_execution = concatenate_raws(raw_files_execution)

            # Datos de imaginación
            # Corregido: Solo pasamos el subject_id y run, y luego el path como parámetro nombrado
            raw_files_imagery = [read_raw_edf(f, preload=True, stim_channel='auto') for f in
                                eegbci.load_data(subjNumber, j, path=DATA_SAMPLE_PATH)]
            raw_imagery = concatenate_raws(raw_files_imagery)

            # Anotaciones para ejecución
            events, _ = mne.events_from_annotations(raw_execution, event_id=dict(T0=1, T1=2, T2=3))
            mapping = {1: 'rest', 2: 'do/feet', 3: 'do/hands'}
            annot_from_events = mne.annotations_from_events(
                events=events, event_desc=mapping, sfreq=raw_execution.info['sfreq'],
                orig_time=raw_execution.info['meas_date'])
            raw_execution.set_annotations(annot_from_events)

            # Anotaciones para imaginación
            events, _ = mne.events_from_annotations(raw_imagery, event_id=dict(T0=1, T1=2, T2=3))
            mapping = {1: 'rest', 2: 'imagine/feet', 3: 'imagine/hands'}
            annot_from_events = mne.annotations_from_events(
                events=events, event_desc=mapping, sfreq=raw_imagery.info['sfreq'],
                orig_time=raw_imagery.info['meas_date'])
            raw_imagery.set_annotations(annot_from_events)

            raw_files.append(raw_execution)
            raw_files.append(raw_imagery)
        except Exception as e:
            print(f"Error al procesar subject {subjNumber}, run {i}/{j}: {str(e)}")
            # Si no hay datos, intenta descargarlos
            try:
                print(f"Intentando descargar datos para subject {subjNumber}...")
                # Corregido: Ajustamos los parámetros para force_update
                eegbci.load_data(subjNumber, i, path=DATA_SAMPLE_PATH, force_update=True)
                eegbci.load_data(subjNumber, j, path=DATA_SAMPLE_PATH, force_update=True)
                print(f"Datos descargados. Por favor, vuelve a ejecutar el script.")
            except Exception as e2:
                print(f"Error al descargar datos: {str(e2)}")
            
    if not raw_files:
        raise ValueError(f"No se pudieron obtener datos para el subject {subjNumber}")
            
    raw = concatenate_raws(raw_files)

    event, event_dict = events_from_annotations(raw)
    picks = pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')

    return [raw, event, event_dict, picks]

def prepare_data(raw, plotIt=False):
    eegbci.standardize(raw)
    montage = make_standard_montage("biosemi64")
    raw.set_montage(montage, on_missing='ignore')

    if plotIt:
        montage = raw.get_montage()
        p = montage.plot()
        p = mne.viz.plot_raw(raw, scalings={"eeg": 75e-6})

    return raw

def filter_data(raw, plotIt=False):
    raw.filter(7, 30, fir_design='firwin', skip_by_annotation='edge')
    if plotIt:
        p = mne.viz.plot_raw(raw, scalings={"eeg": 75e-6})
        plt.show()
    return raw

def filter_eye_artifacts(raw, picks, method, plotIt=False):
    raw_corrected = raw.copy()
    n_components = 20

    ica = ICA(n_components=n_components, method=method, fit_params=None, random_state=97)
    ica.fit(raw_corrected, picks=picks)

    [eog_indicies, scores] = ica.find_bads_eog(raw, ch_name='Fpz', threshold=1.5)
    ica.exclude.extend(eog_indicies)
    ica.apply(raw_corrected, n_pca_components=n_components, exclude=ica.exclude)

    if plotIt:
        ica.plot_components()
        ica.plot_scores(scores, exclude=eog_indicies)
        plt.show()

    return raw_corrected

def fetch_events(data_filtered, tmin=-1., tmax=4.):
    events, event_ids = events_from_annotations(data_filtered)
    picks = mne.pick_types(data_filtered.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')
    epochs = mne.Epochs(data_filtered, events, event_ids, tmin, tmax, proj=True,
                        picks=picks, baseline=None, preload=True)
    labels = epochs.events[:, -1]
    return labels, epochs, picks

def pre_process_data(subjectID, experiments):
    [raw, event, event_dict, picks] = fetch_data(subjectID)
    raw_prepared = prepare_data(raw)
    raw_filtered = filter_data(raw_prepared)
    labels, epochs, picks = fetch_events(raw_filtered)

    # Extraer solo las épocas correspondientes a las etiquetas seleccionadas
    selected_epochs = epochs[experiments]
    X = selected_epochs.get_data()
    y = selected_epochs.events[:, -1] - 1
    
    # Comprobar si hay suficientes datos
    classes, counts = np.unique(y, return_counts=True)
    print(f"Distribución de clases: {dict(zip(classes, counts))}")
    
    min_samples = min(counts)
    if min_samples < 10:
        print(f"⚠️ ADVERTENCIA: Muy pocas muestras para alguna clase (mínimo {min_samples}).")
        print("   Esto puede conducir a sobreajuste. Considere obtener más datos.")
    
    # Comprobar si los datos están balanceados
    if max(counts) > min_samples * 1.5:  # Si la clase mayoritaria tiene 50% más muestras
        print("⚠️ ADVERTENCIA: Datos desbalanceados. Considere técnicas de balanceo.")

    return [X, y, epochs]

# Funciones para entrenamiento
def pipeline_creation(X, y, transformer1, transformer2=None, transformer3=None):
    cv = ShuffleSplit(10, test_size=0.2, random_state=42)

    # Modificar los clasificadores para prevenir el sobreajuste
    # LDA con regularización más fuerte
    lda = LDA(solver='lsqr', shrinkage=0.5)  # Aumentamos la regularización
    
    # LogisticRegression con regularización más fuerte y early stopping
    log_reg = LogisticRegression(
        penalty='l1', 
        solver='saga',  # Cambiado a 'saga' que soporta early stopping
        C=0.5,  # Menor C = mayor regularización
        max_iter=100,  # Limitar número de iteraciones
        tol=1e-3,  # Tolerancia más alta para convergencia
        multi_class='auto'
    )
    
    # RandomForest con menos árboles y más restricciones
    rfc = RandomForestClassifier(
        n_estimators=50,  # Menos árboles para reducir complejidad
        max_depth=5,      # Profundidad limitada
        min_samples_split=5,  # Más muestras requeridas para dividir
        random_state=42
    )

    final_result = []

    # Añadimos un mensaje para advertir de posible sobreajuste
    def check_overfitting(scores):
        if scores.max() > 0.98:  # Si el score máximo es muy alto
            print("⚠️ ADVERTENCIA: Posible sobreajuste detectado (score > 0.98)")
            print("   Considere usar más regularización o aumentar el conjunto de datos.")
        return scores

    pipeline1 = make_pipeline(transformer1, lda)
    scores1 = cross_val_score(pipeline1, X, y, cv=cv, n_jobs=1)
    scores1 = check_overfitting(scores1)
    final_result.append(('LDA ', pipeline1, scores1))
    
    if transformer2:
        pipeline2 = make_pipeline(transformer2, log_reg)
        scores2 = cross_val_score(pipeline2, X, y, cv=cv, n_jobs=1)
        scores2 = check_overfitting(scores2)
        final_result.append(('LOGR', pipeline2, scores2))
    
    if transformer3:
        pipeline3 = make_pipeline(transformer3, rfc)
        scores3 = cross_val_score(pipeline3, X, y, cv=cv, n_jobs=1)
        scores3 = check_overfitting(scores3)
        final_result.append(('RFC', pipeline3, scores3))

    return final_result

def save_pipeline(pipe, epochs_data_train, labels, subjectID, experiment_name):
    # Dividir los datos en entrenamiento y validación para early stopping
    from sklearn.model_selection import train_test_split
    
    X_train, X_val, y_train, y_val = train_test_split(
        epochs_data_train, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    # Primero entrenar con los datos de entrenamiento
    pipe.fit(X_train, y_train)
    
    # Evaluar en los datos de validación
    val_score = pipe.score(X_val, y_val)
    train_score = pipe.score(X_train, y_train)
    
    print(f"Scores para modelo guardado - Train: {train_score:.3f}, Validación: {val_score:.3f}")
    
    # Verificar posible sobreajuste
    if train_score - val_score > 0.15:  # Diferencia significativa entre train y val
        print("⚠️ ADVERTENCIA: Posible sobreajuste (diferencia train-val > 0.15)")
        print("   Se aplica regularización adicional antes de guardar...")
        
        # Si hay sobreajuste, intentar ajustar hiperparámetros del último paso
        # (asumiendo que es un clasificador con un parámetro de regularización)
        try:
            # Si es un clasificador con parámetro C (LogisticRegression)
            if hasattr(pipe.steps[-1][1], 'C'):
                original_C = pipe.steps[-1][1].C
                pipe.steps[-1][1].C = original_C * 0.5  # Aumentar la regularización
                print(f"   Regularización ajustada: C {original_C} -> {pipe.steps[-1][1].C}")
            
            # Si es un clasificador con parámetro alpha (algunos modelos lineales)
            elif hasattr(pipe.steps[-1][1], 'alpha'):
                original_alpha = pipe.steps[-1][1].alpha
                pipe.steps[-1][1].alpha = original_alpha * 2  # Aumentar la regularización
                print(f"   Regularización ajustada: alpha {original_alpha} -> {pipe.steps[-1][1].alpha}")
            
            # Si es RandomForest, reducir la complejidad
            elif hasattr(pipe.steps[-1][1], 'max_depth'):
                if pipe.steps[-1][1].max_depth is not None:
                    original_depth = pipe.steps[-1][1].max_depth
                    pipe.steps[-1][1].max_depth = max(3, original_depth - 2)  # Reducir profundidad
                    print(f"   Complejidad ajustada: max_depth {original_depth} -> {pipe.steps[-1][1].max_depth}")
        except Exception as e:
            print(f"   No se pudo ajustar automáticamente: {str(e)}")
        
        # Reentrenar con todos los datos después de ajustar hiperparámetros
        pipe.fit(epochs_data_train, labels)
    else:
        # Si no hay sobreajuste grave, reentrenar con todos los datos
        pipe.fit(epochs_data_train, labels)
    
    fileName = f"../models/model_subject_{subjectID}_{experiment_name}.joblib"
    dump(pipe, fileName)
    return

def train_data(X, y, transformer="CSP", run_all_pipelines=False):
    if transformer == "CSP":
        from mne.decoding import CSP as MNE_CSP
        # using CSP transformers from MNE
        csp1 = MNE_CSP()

        if run_all_pipelines:
            csp2 = MNE_CSP()
            csp3 = MNE_CSP()
            return pipeline_creation(X, y, csp1, csp2, csp3)
        return pipeline_creation(X, y, csp1)

    elif transformer == "FAST_CSP":
        # using custom CSP transformers
        csp1 = CSP()

        if run_all_pipelines:
            csp2 = CSP()
            csp3 = CSP()
            return pipeline_creation(X, y, csp1, csp2, csp3)
        return pipeline_creation(X, y, csp1)

    elif transformer == "SPoC":
        # using Spoc transformers
        Spoc1 = SPoC(n_components=15, reg='oas', log=True, rank='full')

        if run_all_pipelines:
            Spoc2 = SPoC(n_components=15, reg='oas', log=True, rank='full')
            Spoc3 = SPoC(n_components=15, reg='oas', log=True, rank='full')
            return pipeline_creation(X, y, Spoc1, Spoc2, Spoc3)
        return pipeline_creation(X, y, Spoc1)
    else:
        raise ValueError(f"Unknown transformer, please enter valid one.")

# Funciones para predicción
def predict(X, y, subjectId, experiment_name, log=True):  # Cambiado a True por defecto
    PREDICT_MODEL = f"../models/model_subject_{subjectId}_{experiment_name}.joblib"
    try:
        clf = load(PREDICT_MODEL)
    except FileNotFoundError as e:
        raise Exception(f"File not found: {PREDICT_MODEL}")

    scores = []
    # Siempre mostramos esta información ahora
    print(f"\nPredicciones para el sujeto {subjectId}:")
    print("epoch_nb =  [prediction]    [truth]    equal?")
    print("---------------------------------------------")
    
    correct_predictions = 0
    total_predictions = X.shape[0]
    
    for n in range(total_predictions):
        pred = clf.predict(X[n:n + 1, :, :])[0]
        truth = y[n:n + 1][0]
        # Mejorar la visualización de la igualdad
        is_equal = pred == truth
        if is_equal:
            correct_predictions += 1
            
        equal_symbol = "✓" if is_equal else "✗"
        
        # Siempre mostrar información de predicción
        print(f"epoch_{n:2} =      [{pred}]           [{truth}]      {equal_symbol}")
        
        scores.append(1 - np.abs(pred - y[n:n + 1][0]))
    
    # Agregar un resumen al final
    accuracy = np.mean(scores).round(3)
    print(f"\nAccuracy total: {accuracy:.3f} ({correct_predictions}/{total_predictions} = {int(accuracy * 100)}%)")
    
    # Advertencia de sobreajuste
    if accuracy > 0.95:
        print("\n⚠️ ADVERTENCIA: La precisión es muy alta (>95%), posible sobreajuste.")
        print("   Considere técnicas como:")
        print("   - Usar más datos de entrenamiento")
        print("   - Aumentar la regularización")
        print("   - Reducir la complejidad del modelo")
        print("   - Implementar early stopping en el entrenamiento")
    
    return accuracy

# Funciones para manejo de argumentos
def get_config():
    """Retorna la configuración por defecto con sujetos y experimento seleccionados aleatoriamente"""
    import random
    
    # Seleccionar 6 sujetos aleatorios entre 1 y 109, sin repeticiones
    random_subjects = random.sample(range(1, 110), 6)  # Selecciona 6 números entre 1 y 109 sin repetición
    
    # Seleccionar un experimento aleatorio
    experiment_options = list(EXPERIMENTS.keys())
    random_experiment = random.choice(experiment_options)
    
    print(f"Sujetos seleccionados aleatoriamente: {random_subjects}")
    print(f"Experimento seleccionado aleatoriamente: {random_experiment}")
    print(f"Eventos del experimento: {EXPERIMENTS[random_experiment][0]} vs {EXPERIMENTS[random_experiment][1]}")
    
    config = {
        'SUBJECTS': random_subjects,  # Sujetos aleatorios
        'MODE': 'all',
        'TRANSFORMER': 'FAST_CSP',
        'EXPERIMENT': random_experiment  # Experimento aleatorio
    }
    return config

# Funciones principales
def hash_list_secure(my_list):
    sorted_tuple = tuple(sorted(my_list))
    hash_object = hashlib.sha256(str(sorted_tuple).encode())
    return hash_object.hexdigest()

def process_subject(subjectID, args, isSingleSubject=False):
    start_time_inner = time.time()
    
    print(f"\n=========== Procesando Sujeto {subjectID} ===========")
    
    try:
        # Intentar cargar datos existentes primero
        print(f"Intentando cargar datos para el sujeto {subjectID}...")
        [X, y, epochs] = pre_process_data(subjectID, EXPERIMENTS[args['EXPERIMENT']])
        print(f"Datos para el sujeto {subjectID} cargados correctamente!")
    except Exception as e:
        print(f"Error procesando el sujeto {subjectID}: {str(e)}")
        # Intentar descargar datos automáticamente si no existen
        try:
            print(f"Intentando descargar datos para el sujeto {subjectID}...")
            for run in EXPERIMENTS_IDS['action'] + EXPERIMENTS_IDS['imagery']:
                downloaded_files = eegbci.load_data(subjectID, run, path=DATA_SAMPLE_PATH, force_update=True)
                print(f"  - Descargados archivos para run {run}: {len(downloaded_files)} archivos")
            
            # Reintentar el procesamiento después de la descarga
            print(f"Reintentando cargar datos para el sujeto {subjectID}...")
            [X, y, epochs] = pre_process_data(subjectID, EXPERIMENTS[args['EXPERIMENT']])
            print(f"Datos para el sujeto {subjectID} cargados correctamente en el segundo intento!")
        except Exception as e2:
            print(f"Error fatal con el sujeto {subjectID}: {str(e2)}")
            return {
                'subject_id': subjectID,
                'error': str(e2),
                'pipelines': [],
                'cross_val_score': 0,
                'accuracy': 0,
                'time_cost': time.time() - start_time_inner
            }

    result_inner = [0, 0]
    output = []
    output.append(f"----------------------------------------------[Subject {subjectID}]")
    stats = {
        'subject_id': subjectID,
        'pipelines': [],
        'cross_val_score': 0,
        'accuracy': 0
    }
    
    if args['MODE'] == "train" or args['MODE'] == "all":
        pipelines = train_data(X=X, y=y, transformer=args['TRANSFORMER'], run_all_pipelines=True)
        best_pipeline = {'cross_val_score': -1}

        print(f"\n----- EVALUACIÓN DE PIPELINES PARA SUJETO {subjectID} -----")
        for pipel in pipelines:
            cross_val_score = pipel[2].mean()
            pipeline_name = pipel[0]
            pipeline = pipel[1]
            output.append(f":--- [S{subjectID}] {pipeline_name} cross_val_score : {cross_val_score.round(2)}")
            print(f"Pipeline {pipeline_name}: cross_val_score = {cross_val_score.round(3)}")

            if cross_val_score > best_pipeline['cross_val_score']:
                best_pipeline = {'name': pipeline_name, 'cross_val_score': cross_val_score, 'pipeline': pipeline}
            stats['pipelines'].append((pipeline_name, cross_val_score))
        
        # Resaltar visualmente cuál pipeline se eligió
        print(f"\n✅ PIPELINE ELEGIDO PARA SUJETO {subjectID}: {best_pipeline['name']}")
        print(f"   Score: {best_pipeline['cross_val_score'].round(3)}")
        print(f"   Descripción: {best_pipeline['pipeline']}")
        
        # Añadir información del pipeline elegido a las estadísticas
        stats['best_pipeline'] = best_pipeline['name']
        
        save_pipeline(best_pipeline['pipeline'], X, y, subjectID, args['EXPERIMENT'])
        result_inner[0] = best_pipeline['cross_val_score']
        stats['cross_val_score'] = result_inner[0]
        print(f"----- FIN DE EVALUACIÓN DE PIPELINES PARA SUJETO {subjectID} -----\n")

    if args['MODE'] == "predict" or args['MODE'] == "all":
        # Siempre mostrar resultados detallados de predicción
        print(f"\n----- RESULTADOS DE PREDICCIÓN PARA SUJETO {subjectID} -----")
        prediction_result = predict(X, y, subjectID, args['EXPERIMENT'], log=True)
        output.append(
            f":--- [S{subjectID}] Prediction accurracy: {'{:.2%}'.format(prediction_result).rstrip('0').rstrip('.')}")
        result_inner[1] = prediction_result
        stats['accuracy'] = result_inner[1]
        print(f"----- FIN DE PREDICCIÓN PARA SUJETO {subjectID} -----\n")

    end_time_inner = time.time()
    time_cost_inner = end_time_inner - start_time_inner
    stats['time_cost'] = time_cost_inner
    print(*output, sep="\n")
    print(f":--- [S{subjectID}] time cost: {round(stats['time_cost'], 2)} seconds")
    return stats

def calculate_all_means(cross_val_scores, accuracy_scores, final_stats):
    print("\n----------------------------[Mean Scores for all subjects]----------------------------")
    
    # Mejorado para mostrar un reporte más detallado
    if cross_val_scores:
        mean_cross_val = np.mean(cross_val_scores).round(3)
        std_cross_val = np.std(cross_val_scores).round(3)
        print(f":--- Mean cross_val score : {mean_cross_val} (std: {std_cross_val})")
        final_stats['mean_cross_val_score'] = float(mean_cross_val)
        final_stats['std_cross_val_score'] = float(std_cross_val)
        
    if accuracy_scores:
        mean_accuracy = np.mean(accuracy_scores).round(3)
        std_accuracy = np.std(accuracy_scores).round(3)
        print(f":--- Mean accuracy       : {mean_accuracy} (std: {std_accuracy})")
        final_stats['mean_accuracy'] = float(mean_accuracy)
        final_stats['std_accuracy'] = float(std_accuracy)
        
    # Agregar lista completa de scores por sujeto al reporte final
    print("\n----------------------------[Individual Scores]----------------------------")
    for i, (cv_score, acc_score) in enumerate(zip(cross_val_scores, accuracy_scores)):
        subj_id = final_stats['subjects'][i]['subject_id']
        # Añadir información del pipeline elegido, si está disponible
        pipeline_info = ""
        if 'best_pipeline' in final_stats['subjects'][i]:
            pipeline_info = f", Pipeline: {final_stats['subjects'][i]['best_pipeline']}"
        print(f"Subject {subj_id}: Cross-val score = {cv_score.round(3)}, Accuracy = {acc_score.round(3)}{pipeline_info}")
    
    # Mostrar resumen de pipelines elegidos
    if any('best_pipeline' in subject for subject in final_stats['subjects']):
        # Contar cuántas veces se eligió cada pipeline
        pipeline_counts = {}
        for subject in final_stats['subjects']:
            if 'best_pipeline' in subject:
                pipeline_name = subject['best_pipeline']
                pipeline_counts[pipeline_name] = pipeline_counts.get(pipeline_name, 0) + 1
        
        print("\n----------------------------[Pipeline Selection Summary]----------------------------")
        for pipeline, count in pipeline_counts.items():
            print(f"Pipeline {pipeline}: elegido {count} veces ({(count/len(final_stats['subjects'])*100):.1f}%)")

def dump_result_to_json(final_stats, args):
    # Crear directorio si no existe
    os.makedirs("../results", exist_ok=True)
    
    # Generar un nombre de archivo más claro y con timestamp
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    results_filename = \
        f"../results/results-{args['MODE']}-{args['EXPERIMENT']}-{timestamp}-{args['TRANSFORMER']}-subjects_{final_stats['subjects_hash']}.json"

    with open(results_filename, 'w', encoding='utf-8') as f:
        json.dump(final_stats, f, ensure_ascii=False, indent=4)

    # Contar qué pipelines fueron seleccionados
    pipeline_counts = {}
    for subject in final_stats['subjects']:
        if 'best_pipeline' in subject:
            pipeline_name = subject['best_pipeline']
            pipeline_counts[pipeline_name] = pipeline_counts.get(pipeline_name, 0) + 1
    
    # Encontrar el pipeline más elegido
    best_pipeline = None
    if pipeline_counts:
        best_pipeline = max(pipeline_counts, key=pipeline_counts.get)

    # Generar y mostrar un resumen más completo
    print("\n===================== RESUMEN FINAL =====================")
    print(f"Experimento: {args['EXPERIMENT']} - Transformer: {args['TRANSFORMER']}")
    print(f"Eventos comparados: {EXPERIMENTS[args['EXPERIMENT']][0]} vs {EXPERIMENTS[args['EXPERIMENT']][1]}")
    print(f"Sujetos analizados: {args['SUBJECTS']}")
    print(f"Sujetos procesados correctamente: {final_stats.get('successful_subjects', 0)}/{len(args['SUBJECTS'])}")
    
    if best_pipeline:
        print(f"Pipeline más utilizado: {best_pipeline} ({pipeline_counts[best_pipeline]} sujetos)")
    
    if 'mean_cross_val_score' in final_stats:
        print(f"Score validación cruzada promedio: {final_stats['mean_cross_val_score']:.3f} (std: {final_stats.get('std_cross_val_score', 0):.3f})")
    if 'mean_accuracy' in final_stats:
        print(f"Precisión de predicción promedio: {final_stats['mean_accuracy']:.3f} (std: {final_stats.get('std_accuracy', 0):.3f})")
    
    print(f"Tiempo total: {round(final_stats['time_cost'], 2)} segundos")
    print(f"Los resultados completos se han guardado en:\n[{results_filename}]")
    
    # Tabla de resumen por sujeto
    print("\n----------- Resumen por Sujeto -----------")
    print("Sujeto | Pipeline | Cross-Val | Accuracy")
    print("----------------------------------------")
    
    # Calcular promedios para la fila final
    total_subjects = 0
    sum_cv_score = 0
    sum_accuracy = 0
    
    for subject in final_stats['subjects']:
        subject_id = subject['subject_id']
        pipeline = subject.get('best_pipeline', 'N/A')
        cv_score = subject.get('cross_val_score', 0)
        accuracy = subject.get('accuracy', 0)
        
        # Solo contar sujetos válidos para el promedio
        if 'error' not in subject:
            total_subjects += 1
            sum_cv_score += cv_score
            sum_accuracy += accuracy
            
        print(f"{subject_id:6} | {pipeline:8} | {cv_score:.3f}    | {accuracy:.3f}")
    
    # Añadir línea divisoria
    print("----------------------------------------")
    
    # Añadir fila con el promedio de Accuracy
    avg_cv = sum_cv_score / total_subjects if total_subjects > 0 else 0
    avg_accuracy = sum_accuracy / total_subjects if total_subjects > 0 else 0
    print(f"PROMEDIO|          | {avg_cv:.3f}    | {avg_accuracy:.3f}")
    
    print("==========================================================")

def ensure_directories():
    """Asegura que los directorios necesarios existan"""
    
    # Directorios necesarios
    dirs = [
        "../data/raw/files",
        "../models",
        "../results",
    ]
    
    for directory in dirs:
        os.makedirs(directory, exist_ok=True)

def main():
    print("\n===================================================================")
    print("           ANÁLISIS DE DATOS EEG CON CLASIFICACIÓN BCI              ")
    print("===================================================================\n")
    
    start_time = time.time()
    
    # Asegurar que los directorios necesarios existan
    ensure_directories()
    
    # Usar configuración predeterminada
    args = get_config()
    print("Configuración:", args)

    print(
        f"Experimento en estudio: ({EXPERIMENTS[args['EXPERIMENT']][0]}) <--VS--> ({EXPERIMENTS[args['EXPERIMENT']][1]})")
    print(f"Transformador: {args['TRANSFORMER']}")
    print(f"Modo: {args['MODE']}")
    print(f"Sujetos a procesar: {args['SUBJECTS']}")
    print("\n===================================================================\n")
    
    CALC_MEAN_FOR_ALL = True if len(args['SUBJECTS']) > 1 else False

    cross_val_scores = []
    accuracy_scores = []
    final_stats = {
        'subjects_hash': "all" if len(args['SUBJECTS']) == 109 else ''.join(map(str, args['SUBJECTS'])),
        'config': args,
        'events': EXPERIMENTS[args['EXPERIMENT']],
        'subjects': [],
        'time_unit': "seconds",
        'total_subjects': len(args['SUBJECTS']),
        'successful_subjects': 0
    }

    for subjectID in args['SUBJECTS']:
        result = process_subject(subjectID, args, isSingleSubject=not CALC_MEAN_FOR_ALL)
        
        # Solo agregar resultados válidos
        if 'error' not in result:
            if args['MODE'] == "train" or args['MODE'] == "all":
                cross_val_scores.append(result['cross_val_score'])

            if args['MODE'] == "predict" or args['MODE'] == "all":
                accuracy_scores.append(result['accuracy'])
                
            final_stats['successful_subjects'] += 1

        final_stats['subjects'].append(result)
        
        # Mostrar progreso
        print(f"\n--- Progreso: {final_stats['subjects'].index(result)+1}/{len(args['SUBJECTS'])} sujetos procesados ---\n")

    if CALC_MEAN_FOR_ALL and cross_val_scores:  # Solo calcular media si hay resultados válidos
        calculate_all_means(cross_val_scores, accuracy_scores, final_stats)

    final_stats['time_cost'] = time.time() - start_time
    print(f":--- Tiempo total de ejecución: {round(final_stats['time_cost'], 2)} segundos")

    dump_result_to_json(final_stats, args)

if __name__ == "__main__":
    main()



           ANÁLISIS DE DATOS EEG CON CLASIFICACIÓN BCI              

Sujetos seleccionados aleatoriamente: [14, 69, 86, 21, 40, 19]
Experimento seleccionado aleatoriamente: hands_vs_feet__imagery
Eventos del experimento: imagine/hands vs imagine/feet
Configuración: {'SUBJECTS': [14, 69, 86, 21, 40, 19], 'MODE': 'all', 'TRANSFORMER': 'FAST_CSP', 'EXPERIMENT': 'hands_vs_feet__imagery'}
Experimento en estudio: (imagine/hands) <--VS--> (imagine/feet)
Transformador: FAST_CSP
Modo: all
Sujetos a procesar: [14, 69, 86, 21, 40, 19]



=========== Procesando Sujeto 14 ===========
Intentando cargar datos para el sujeto 14...


Distribución de clases: {2: 24, 3: 21}
Datos para el sujeto 14 cargados correctamente!

----- EVALUACIÓN DE PIPELINES PARA SUJETO 14 -----
Pipeline LDA : cross_val_score = 0.378
Pipeline LOGR: cross_val_score = 0.422
Pipeline RFC: cross_val_score = 0.433

✅ PIPELINE ELEGIDO PARA SUJETO 14: RFC
   Score: 0.433
   Descripción: Pipeline(steps=[('csp', CSP()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=5, min_samples_split=5,
                                        n_estimators=50, random_state=42))])
Scores para modelo guardado - Train: 0.972, Validación: 0.556
⚠️ ADVERTENCIA: Posible sobreajuste (diferencia train-val > 0.15)
   Se aplica regularización adicional antes de guardar...
   Complejidad ajustada: max_depth 5 -> 3
----- FIN DE EVALUACIÓN DE PIPELINES PARA SUJETO 14 -----


----- RESULTADOS DE PREDICCIÓN PARA SUJETO 14 -----

Predicciones para el sujeto 14:
epoch_nb =  [prediction]    [truth]    equal?
----------------------------

Distribución de clases: {2: 22, 3: 23}
Datos para el sujeto 69 cargados correctamente!


/sgoinfre/students/dgerwig-/miniforge/envs/vortex_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/sgoinfre/students/dgerwig-/miniforge/envs/vortex_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/sgoinfre/students/dgerwig-/miniforge/envs/vortex_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



----- EVALUACIÓN DE PIPELINES PARA SUJETO 69 -----
Pipeline LDA : cross_val_score = 0.667
Pipeline LOGR: cross_val_score = 0.644
Pipeline RFC: cross_val_score = 0.678

✅ PIPELINE ELEGIDO PARA SUJETO 69: RFC
   Score: 0.678
   Descripción: Pipeline(steps=[('csp', CSP()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=5, min_samples_split=5,
                                        n_estimators=50, random_state=42))])
Scores para modelo guardado - Train: 0.944, Validación: 0.556
⚠️ ADVERTENCIA: Posible sobreajuste (diferencia train-val > 0.15)
   Se aplica regularización adicional antes de guardar...
   Complejidad ajustada: max_depth 5 -> 3
----- FIN DE EVALUACIÓN DE PIPELINES PARA SUJETO 69 -----


----- RESULTADOS DE PREDICCIÓN PARA SUJETO 69 -----

Predicciones para el sujeto 69:
epoch_nb =  [prediction]    [truth]    equal?
---------------------------------------------
epoch_ 0 =      [3]           [3]      ✓
epoch_ 1 =      [2]         

...

Distribución de clases: {2: 22, 3: 23}
Datos para el sujeto 40 cargados correctamente!
⚠️ ADVERTENCIA: Posible sobreajuste detectado (score > 0.98)
   Considere usar más regularización o aumentar el conjunto de datos.
⚠️ ADVERTENCIA: Posible sobreajuste detectado (score > 0.98)
   Considere usar más regularización o aumentar el conjunto de datos.

----- EVALUACIÓN DE PIPELINES PARA SUJETO 40 -----
Pipeline LDA : cross_val_score = 0.778
Pipeline LOGR: cross_val_score = 0.789
Pipeline RFC: cross_val_score = 0.711

✅ PIPELINE ELEGIDO PARA SUJETO 40: LOGR
   Score: 0.789
   Descripción: Pipeline(steps=[('csp', CSP()),
                ('logisticregression',
                 LogisticRegression(C=0.5, penalty='l1', solver='saga',
                                    tol=0.001))])
Scores para modelo guardado - Train: 1.000, Validación: 0.778
⚠️ ADVERTENCIA: Posible sobreajuste (diferencia train-val > 0.15)
   Se aplica regularización adicional antes de guardar...
   Regularización ajustada: C 0

----- FIN DE EVALUACIÓN DE PIPELINES PARA SUJETO 40 -----


----- RESULTADOS DE PREDICCIÓN PARA SUJETO 40 -----

Predicciones para el sujeto 40:
epoch_nb =  [prediction]    [truth]    equal?
---------------------------------------------
epoch_ 0 =      [2]           [2]      ✓
epoch_ 1 =      [3]           [3]      ✓
epoch_ 2 =      [3]           [3]      ✓
epoch_ 3 =      [2]           [2]      ✓
epoch_ 4 =      [2]           [2]      ✓
epoch_ 5 =      [3]           [3]      ✓
epoch_ 6 =      [2]           [2]      ✓
epoch_ 7 =      [3]           [3]      ✓
epoch_ 8 =      [3]           [3]      ✓
epoch_ 9 =      [2]           [2]      ✓
epoch_10 =      [2]           [2]      ✓
epoch_11 =      [3]           [3]      ✓
epoch_12 =      [3]           [3]      ✓
epoch_13 =      [2]           [2]      ✓
epoch_14 =      [2]           [2]      ✓
epoch_15 =      [3]           [3]      ✓
epoch_16 =      [2]           [2]      ✓
epoch_17 =      [2]           [2]      ✓
epoch_18 =      [3]      

Distribución de clases: {2: 22, 3: 23}
Datos para el sujeto 19 cargados correctamente!


/sgoinfre/students/dgerwig-/miniforge/envs/vortex_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/sgoinfre/students/dgerwig-/miniforge/envs/vortex_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



----- EVALUACIÓN DE PIPELINES PARA SUJETO 19 -----
Pipeline LDA : cross_val_score = 0.544
Pipeline LOGR: cross_val_score = 0.522
Pipeline RFC: cross_val_score = 0.556

✅ PIPELINE ELEGIDO PARA SUJETO 19: RFC
   Score: 0.556
   Descripción: Pipeline(steps=[('csp', CSP()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=5, min_samples_split=5,
                                        n_estimators=50, random_state=42))])
Scores para modelo guardado - Train: 0.917, Validación: 0.667
⚠️ ADVERTENCIA: Posible sobreajuste (diferencia train-val > 0.15)
   Se aplica regularización adicional antes de guardar...
   Complejidad ajustada: max_depth 5 -> 3
----- FIN DE EVALUACIÓN DE PIPELINES PARA SUJETO 19 -----


----- RESULTADOS DE PREDICCIÓN PARA SUJETO 19 -----

Predicciones para el sujeto 19:
epoch_nb =  [prediction]    [truth]    equal?
---------------------------------------------
epoch_ 0 =      [2]           [2]      ✓
epoch_ 1 =      [3]         